# 03 - Preprocessing and Datasets Split

## Data sources
- **ML-HydPARK** (Zenodo, v0.0.5): cleaned experimental thermodynamic data for metal hydrides (~770 entries)
- **ElementalH_Ef** and **ElementalH_Ef_MP**: reserved for compositional feature engineering (future step)

## Target Variables
- `Temperature_oC`
- `Hydrogen_Weight_Percent`

## Goals
- Handling missing values
- Feature selection and engineering
- Split into Dataset A and Dataset B
- Per-dataset: encoding, scaling and train/val/test split

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import os

In [2]:
# Loading the dataset
df = pd.read_csv('../data/processed/ML-HYDPARK_eda_cleaned.csv')
df.shape

(772, 9)

In [3]:
df.head()

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK,LnEquilibrium_Pressure_25C,HtoM
0,A2B,Th2Al,0.8,130.0,500.0,0.001,110.711134,-39.127349,1.309571
1,A2B,Ti2Cu,2.2,130.0,500.0,0.120,150.515102,-34.339857,1.184850
2,A2B,Zr2Cu,1.3,144.0,600.0,0.003,116.621978,-44.064155,1.071443
3,A2B,Zr2Ni,1.3,183.0,604.0,0.003,160.332084,-54.539843,1.050307
4,A2B,Mg2Ni,3.6,64.5,299.0,3.200,122.403296,-11.297687,1.325126


## Dropping of features

`LnEquilibrium_Pressure_25C`: it is redundant as it can be calculated from `Heat_of_Formation_kJperMolH2` and `Entropy_of_Formation_JperMolH2perK` via the van 't Hoff equation.

`HtoM`: it is mathematically derivable from `Hydrogen_Weight_Percent`, so keeping both when one is the target variable would introduce data leakage.

In [4]:
# Dropping 'LnEquilibrium_Pressure_25C'
df = df.drop(columns = ['LnEquilibrium_Pressure_25C'], axis = 1)
df.shape

(772, 8)

In [5]:
# Dropping 'HtoM'
df = df.drop(columns = ['HtoM'], axis = 1)
df.shape

(772, 7)

In [6]:
df.head()

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK
0,A2B,Th2Al,0.8,130.0,500.0,0.001,110.711134
1,A2B,Ti2Cu,2.2,130.0,500.0,0.120,150.515102
2,A2B,Zr2Cu,1.3,144.0,600.0,0.003,116.621978
3,A2B,Zr2Ni,1.3,183.0,604.0,0.003,160.332084
4,A2B,Mg2Ni,3.6,64.5,299.0,3.200,122.403296


## Handling Missing Values

In [7]:
# Checking missing values across the cleaned dataset
df.isna().sum()

Material_Class                         25
Composition_Formula                     0
Hydrogen_Weight_Percent                31
Heat_of_Formation_kJperMolH2            0
Temperature_oC                        360
Pressure_Atmospheres_Absolute         369
Entropy_of_Formation_JperMolH2perK      0
dtype: int64

In [8]:
# Further exploring the dataset in order to impute the missing values for 'Material_Class'
df.sample(20)

,Material_Class,Composition_Formula,Hydrogen_Weight_Percent,Heat_of_Formation_kJperMolH2,Temperature_oC,Pressure_Atmospheres_Absolute,Entropy_of_Formation_JperMolH2perK
633,AB2(C14),(Ti0.9Zr0.1)(V0.1Cr0.35Mn0.55)2,1.882036,22.92,NaN,NaN,97.900000
314,MIC,Ho2Co7,1.100000,36.70,50.0,5.00,126.950672
725,AB2(C15),(Ti0.8Zr0.2)(V0.1Fe0.5Ni0.4)2,2.146834,15.10,NaN,NaN,117.000000
16,AB,Zr0.1Ti0.9Fe1,1.000000,28.90,30.0,3.30,105.258805
27,AB,Ti1Mn0.05Fe0.95,1.600000,29.30,50.0,9.00,108.938042
582,AB2(C14),(Ti0.85Zr0.15)(Cr0.45Mn0.5Mo0.05)1.82,1.775446,23.70,NaN,NaN,106.400000
692,AB2(C14),(Zr)(Mn0.71Ni0.29)2.8,1.295236,18.60,NaN,NaN,70.800000
177,AB5,CeNi5,1.500000,22.20,23.0,80.00,111.394879
252,AB5,NdCo5,0.620000,42.70,22.0,0.70,141.706748
650,AB2(C14),(Zr)(Cr0.3Fe0.7)2,1.484969,28.50,NaN,NaN,94.700000


### Imputing `Material_Class`

25 entries have no `Material_Class` label. Rather than dropping them, we assign classes manually by inspecting the `Composition_Formula` and applying the stoichiometric A:B ratio rule:
- Group elements into A-site (larger atoms) and B-site (smaller transition metals)
- The total A:B atom count determines the class (1:5 → AB5, 1:2 → AB2, 1:1 → AB)
- V-based alloys with no clean integer ratio are solid solutions (SS)
- AB2 subtype (C14 hexagonal vs C15 cubic Laves) is assigned based on B-site chemistry: Mn-dominated → C14, Fe/Cr-dominated with ZrFe2 or TiCr2 parent → C15

Classes were verified against existing entries in the dataset and cross-checked with literature on Laves phase stability.

In [9]:
# Manual imputation of Material_Class for 25 entries with missing labels
# Assignment based on A:B stoichiometric ratio and Laves phase B-site chemistry
class_map = {
    # AB5 - La/Ce/Y on A-site, Ni+substituents sum to 5 on B-site
    'LaNi4.7Sn0.3':          'AB5',
    'LaNi4.8Sn0.2':          'AB5',
    'LaNi4.8Al0.2':          'AB5',
    'La0.85Ce0.15Ni5':       'AB5',
    'La0.2Y0.8Ni4.6Mn0.4':  'AB5',
    'La0.4Ce0.4Ca0.2Ni5':   'AB5',
    # AB - 1:1 ratio
    'TiFe0.9Mn0.1':          'AB',
    # SS - V-based BCC solid solutions, no ordered A/B sublattice
    'V75Ti17.5Zr7.5':        'SS',
    'V75Ti10Zr7.5Cr7.5':     'SS',
    'V92.5Zr7.5':            'SS',
    'V0.85Ti0.1Fe0.05':      'SS',
    # AB2(C15) - TiCr2 or ZrFe2 parent, Cr/Fe-dominated B-site, cubic Laves
    'TiCr1.9':                        'AB2(C15)',
    'TiCr1.9Mo0.01':                  'AB2(C15)',
    'Ti0.86Mo0.14Cr1.9':              'AB2(C15)',
    'ZrFe1.8Cr0.2':                   'AB2(C15)',
    'ZrFe1.8Ni0.2':                   'AB2(C15)',
    'Zr0.8Ti0.2FeNi0.8V0.2':         'AB2(C15)',
    # AB2(C14) - Mn-dominated B-site or significant Mn substitution, hexagonal Laves
    'TiCrMn':                                          'AB2(C14)',
    'Ti0.8Zr0.2CrMn':                                 'AB2(C14)',
    'TiCr1.5Mn0.25Fe0.25':                            'AB2(C14)',
    'TiCr1.5Mn0.2Fe0.3':                              'AB2(C14)',
    '(Ti0.97Zr0.03)1.1Cr1.6Mn0.4':                   'AB2(C14)',
    'Zr0.7Ti0.3Mn2':                                  'AB2(C14)',
    'Ti0.9Zr0.1Mn1.4Cr0.35V0.2Fe0.05':               'AB2(C14)',
    'Ti0.77Zr0.3Cr0.85Fe0.7Mn0.25Ni0.2Cu0.03':       'AB2(C14)',
}

for formula, cls in class_map.items():
    df.loc[df['Composition_Formula'] == formula, 'Material_Class'] = cls

df['Material_Class'].isna().sum()

np.int64(0)

## Dataset Split

In [10]:
# Splitting the dataset into Dataset A and Dataset B
# Target variables are 'Hydrogen_Weight_Percent' and 'Temperature_oC' respectively
df_A = df.dropna(subset=['Hydrogen_Weight_Percent']).copy()
df_B = df.dropna(subset=['Temperature_oC']).copy()

print(df_A.shape)
print(df_B.shape)

(741, 7)
(412, 7)


In [11]:
# Checking NaNs in each dataset
print(df_A.isna().sum())
print("---")
print(df_B.isna().sum())

Material_Class                          0
Composition_Formula                     0
Hydrogen_Weight_Percent                 0
Heat_of_Formation_kJperMolH2            0
Temperature_oC                        351
Pressure_Atmospheres_Absolute         360
Entropy_of_Formation_JperMolH2perK      0
dtype: int64
---
Material_Class                         0
Composition_Formula                    0
Hydrogen_Weight_Percent               22
Heat_of_Formation_kJperMolH2           0
Temperature_oC                         0
Pressure_Atmospheres_Absolute          9
Entropy_of_Formation_JperMolH2perK     0
dtype: int64


### Handling remaining NaNs per dataset

**Dataset A** (`Hydrogen_Weight_Percent` as target):
- `Temperature_oC`: 351/741 = 47% missing. Too sparse to impute reliably, column will be dropped. It is also the target variable of Dataset B, so using it as a feature here would be conceptually problematic.
- `Pressure_Atmospheres_Absolute`: 360/741 = 49% missing. Same reasoning, will be dropped.

**Dataset B** (`Temperature_oC` as target):
- `Hydrogen_Weight_Percent`: 22/412 = 5% missing. Rows to be dropped.
- `Pressure_Atmospheres_Absolute`: 9/412 = 2% missing. Rows to be dropped. Column subsequently dropped entirely due to target leakage (see below).

In [12]:
# Dropping 'Temperature_oC' and 'Pressure_Atmospheres_Absolute' from df_A
df_A = df_A.drop(columns = ['Temperature_oC', 'Pressure_Atmospheres_Absolute'], axis = 1)
df_A.shape

(741, 5)

In [13]:
# Dropping rows where Hydrogen_Weight_Percent or Pressure_Atmospheres_Absolute is null in df_B
df_B = df_B.dropna(subset=['Hydrogen_Weight_Percent', 'Pressure_Atmospheres_Absolute'])
df_B.shape

(381, 7)

In [14]:
# Dropping 'Pressure_Atmospheres_Absolute' from df_B
# Leaky feature: at equilibrium, P, T, ΔH and ΔS are linked by van 't Hoff
# The model would learn the equation rather than the underlying chemistry
df_B = df_B.drop(columns=['Pressure_Atmospheres_Absolute'])
df_B.shape


(381, 6)

### Cleaned Datasets Overview

Dataset A: 741 rows, 5 columns. Target: `Hydrogen_Weight_Percent`

Dataset B: 381 rows, 6 columns. Target: `Temperature_oC`

### One-hot encoding of categorical features

One-hot encoding applied to `Material_Class` in both Dataset A and Dataset B.

`Composition_Formula` is not encoded. It contains ~700 unique string values, and one-hot encoding would create hundreds of uninformative columns. It will be examined during compositional feature engineering (future step).

In [15]:
df_A.info()

<class 'pandas.core.frame.DataFrame'>
Index: 741 entries, 0 to 771
Data columns (total 5 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Material_Class                      741 non-null    object 
 1   Composition_Formula                 741 non-null    object 
 2   Hydrogen_Weight_Percent             741 non-null    float64
 3   Heat_of_Formation_kJperMolH2        741 non-null    float64
 4   Entropy_of_Formation_JperMolH2perK  741 non-null    float64
dtypes: float64(3), object(2)
memory usage: 34.7+ KB


In [16]:
# One-hot encoding Material_Class in Dataset A
df_A = pd.get_dummies(df_A, columns=['Material_Class'])
df_A.shape

(741, 13)

In [17]:
df_B.info()

<class 'pandas.core.frame.DataFrame'>
Index: 381 entries, 0 to 427
Data columns (total 6 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Material_Class                      381 non-null    object 
 1   Composition_Formula                 381 non-null    object 
 2   Hydrogen_Weight_Percent             381 non-null    float64
 3   Heat_of_Formation_kJperMolH2        381 non-null    float64
 4   Temperature_oC                      381 non-null    float64
 5   Entropy_of_Formation_JperMolH2perK  381 non-null    float64
dtypes: float64(4), object(2)
memory usage: 20.8+ KB


In [18]:
# One-hot encoding Material_Class in Dataset B
df_B = pd.get_dummies(df_B, columns=['Material_Class'])
df_B.shape

(381, 12)

In [19]:
# Dropping Composition_Formula: string column, therefore not used as model input
df_A = df_A.drop(columns=['Composition_Formula'])
df_B = df_B.drop(columns=['Composition_Formula'])
print(df_A.shape)
print(df_B.shape)


(741, 12)
(381, 11)


## Train/Val/Test Split

Each dataset is split into train (70%), validation (15%), and test (15%) sets. The split is performed in two steps: first a 70/30 split, then the 30% is halved into val and test. `random_state=42` ensures reproducibility.

In [20]:
# Splitting dataset A in target variable and features
X_A = df_A.drop(columns=['Hydrogen_Weight_Percent'])
y_A = df_A['Hydrogen_Weight_Percent']

# Splitting dataset B in target variable and features
X_B = df_B.drop(columns=['Temperature_oC'])
y_B = df_B['Temperature_oC']


In [21]:
# Train/val/test split for Dataset A (70/15/15)
X_A_train, X_A_temp, y_A_train, y_A_temp = train_test_split(X_A, y_A, test_size=0.30, random_state=42)
X_A_val, X_A_test, y_A_val, y_A_test = train_test_split(X_A_temp, y_A_temp, test_size=0.50, random_state=42)

print(X_A_train.shape, X_A_val.shape, X_A_test.shape)


(518, 11) (111, 11) (112, 11)


In [22]:
# Train/val/test split for Dataset B (70/15/15)
X_B_train, X_B_temp, y_B_train, y_B_temp = train_test_split(X_B, y_B, test_size=0.30, random_state=42)
X_B_val, X_B_test, y_B_val, y_B_test = train_test_split(X_B_temp, y_B_temp, test_size=0.50, random_state=42)

print(X_B_train.shape, X_B_val.shape, X_B_test.shape)


(266, 10) (57, 10) (58, 10)


## Scaling

RobustScaler is used as it is more robust to outliers: it scales data using the median and IQR, ignoring extreme values. This is appropriate here as maximum and minimum values may represent legitimate experimental conditions.

The scaler is fitted on the training set only, then applied to validation and test sets without refitting. Fitting on val or test would leak information about those sets into the scaling parameters, constituting data leakage.

A separate scaler is fitted for Dataset A and Dataset B.

In [ ]:
# Scaling all numeric features except each dataset's target variable so that they are comparable
numeric_cols_A = ['Heat_of_Formation_kJperMolH2', 'Entropy_of_Formation_JperMolH2perK']
numeric_cols_B = ['Heat_of_Formation_kJperMolH2', 'Entropy_of_Formation_JperMolH2perK', 'Hydrogen_Weight_Percent']

In [24]:
# Fit RobustScaler on X_A_train only, then apply to val and test
scaler_A = RobustScaler()
X_A_train[numeric_cols_A] = scaler_A.fit_transform(X_A_train[numeric_cols_A])
X_A_val[numeric_cols_A] = scaler_A.transform(X_A_val[numeric_cols_A])
X_A_test[numeric_cols_A] = scaler_A.transform(X_A_test[numeric_cols_A])


In [25]:
# Fit RobustScaler on X_B_train only, then apply to val and test
scaler_B = RobustScaler()
X_B_train[numeric_cols_B] = scaler_B.fit_transform(X_B_train[numeric_cols_B])
X_B_val[numeric_cols_B] = scaler_B.transform(X_B_val[numeric_cols_B])
X_B_test[numeric_cols_B] = scaler_B.transform(X_B_test[numeric_cols_B])

## Saving the split datasets

In [26]:
# Creating the folder where to save the split datasets
os.makedirs('../data/processed/splits', exist_ok=True)

# Dataset A
X_A_train.to_csv('../data/processed/splits/X_A_train.csv', index=False)
X_A_val.to_csv('../data/processed/splits/X_A_val.csv', index=False)
X_A_test.to_csv('../data/processed/splits/X_A_test.csv', index=False)
y_A_train.to_csv('../data/processed/splits/y_A_train.csv', index=False)
y_A_val.to_csv('../data/processed/splits/y_A_val.csv', index=False)
y_A_test.to_csv('../data/processed/splits/y_A_test.csv', index=False)

# Dataset B
X_B_train.to_csv('../data/processed/splits/X_B_train.csv', index=False)
X_B_val.to_csv('../data/processed/splits/X_B_val.csv', index=False)
X_B_test.to_csv('../data/processed/splits/X_B_test.csv', index=False)
y_B_train.to_csv('../data/processed/splits/y_B_train.csv', index=False)
y_B_val.to_csv('../data/processed/splits/y_B_val.csv', index=False)
y_B_test.to_csv('../data/processed/splits/y_B_test.csv', index=False)

print('Splits saved.')


Splits saved.


## Final Datasets Overview

**Dataset A**

Target: `Hydrogen_Weight_Percent`
- Train: 518 samples, 11 features
- Val: 111 samples
- Test: 112 samples

**Dataset B**

Target: `Temperature_oC`
- Train: 266 samples, 10 features
- Val: 57 samples
- Test: 58 samples